### Building a RAG System with LangChain and FAISS 
Introduction to RAG (Retrieval-Augmented Generation)
RAG combines the power of retrieval systems with generative AI models. Instead of relying solely on the model's training data, RAG:

1. Retrieves relevant documents from a knowledge base
2. Uses these documents as context for the LLM
3. Generates responses based on both the retrieved context and the model's knowledge

### FAISS 
https://github.com/facebookresearch/faiss

FAISS is a library for efficient similarity search and clustering of dense vectors.

Key advantages:
1. Extremely fast similarity search
2. Memory efficient
3. Supports GPU acceleration
4. Can handle millions of vectors

How it works:
- Indexes vectors for fast nearest neighbor search
- Returns most similar vectors based on distance metrics


In [1]:
import os
from dotenv import load_dotenv
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import AIMessage, HumanMessage

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

load_dotenv()


True

### Data Ingestion And Processing

In [2]:
sample_documents = [
    Document(
        page_content="""
        Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
        """,
        metadata={"source": "AI Introduction", "page": 1, "topic": "AI"}
    ),
    Document(
        page_content="""
        Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.
        """,
        metadata={"source": "ML Basics", "page": 1, "topic": "ML"}
    ),
    Document(
        page_content="""
        Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition.
        """,
        metadata={"source": "Deep Learning", "page": 1, "topic": "DL"}
    ),
    Document(
        page_content="""
        Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
        Applications include chatbots, translation, sentiment analysis, and text summarization.
        """,
        metadata={"source": "NLP Overview", "page": 1, "topic": "NLP"}
    )
]

print(sample_documents)

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='\n        Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.\n        '), Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='\n        Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.\n        '), Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='\n        Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolu

In [3]:
### Text Splitting
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    length_function = len,
    separators = [" "]
)
chunks = text_splitter.split_documents(sample_documents)
print(chunks[0])
print(chunks[1])

page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.' metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}
page_content='Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.' metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}


In [4]:

print(f"Created {len(chunks)} chunks from {len(sample_documents)} documents")
print("\nExample chunk:")
print(f"Content: {chunks[0].page_content}")
print(f"Metadata: {chunks[0].metadata}")

Created 4 chunks from 4 documents

Example chunk:
Content: Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
Metadata: {'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}


In [5]:
## load the embedding models

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_BASE_URL")
embeddings = OpenAIEmbeddings(
    model = "text-embedding-3-small",
    # dimensions = 1536,
    api_key = api_key,
    base_url = base_url
)

sample_text = "What is machine learning?"
sample_embedding = embeddings.embed_query(sample_text)
print(f"Sample embedding shape: {len(sample_embedding)}")
print(f"Sample embedding: {sample_embedding}")

Sample embedding shape: 1536
Sample embedding: [-0.002476818859577179, -0.012755980715155602, -0.006645360495895147, -0.03157883137464523, 0.028759293258190155, 0.019133972004055977, -0.004931761883199215, 0.001355079934000969, -0.022342411801218987, 0.04985721781849861, 0.010296177119016647, -0.017451971769332886, -0.03881240636110306, 0.0008811057778075337, 0.046201542019844055, 0.016946399584412575, -0.00980518851429224, -0.018404779955744743, 0.015497739426791668, 0.0040931920520961285, 0.017549196258187294, 0.04005689173936844, 0.056351881474256516, -0.032337188720703125, 0.023898020386695862, -0.03297887742519379, 0.019338145852088928, 0.011647610925137997, -0.006854395382106304, -0.010850362479686737, 0.005911308340728283, -0.019260365515947342, -0.02868151292204857, 0.024948054924607277, 0.02597864530980587, -0.004068885929882526, -0.016178317368030548, -0.011637887917459011, -0.051996178925037384, 0.016596386209130287, -0.05779081583023071, -0.00763706024736166, 0.012182351201

In [6]:
texts = ["AI", "Machine Learning", "Deep Learning", "Neural Networks"]
batch_embeddings = embeddings.embed_documents(texts)
print(batch_embeddings[0])

[-0.0081634521484375, -0.0246124267578125, 0.0029850006103515625, 0.0251617431640625, 0.006565093994140625, -0.028228759765625, -0.005023956298828125, 0.020904541015625, -0.036895751953125, 0.01279449462890625, -0.0030364990234375, -0.020111083984375, 0.0002522468566894531, -0.03271484375, 0.0064544677734375, -0.0252685546875, -0.031097412109375, -0.054412841796875, 0.03277587890625, -0.0184173583984375, 0.01666259765625, 0.04833984375, -0.024871826171875, 0.01438140869140625, 0.0293426513671875, 0.004047393798828125, 0.00928497314453125, 0.01337432861328125, 0.0025310516357421875, -0.0225372314453125, 0.0321044921875, -0.0280303955078125, 0.005359649658203125, -0.038177490234375, -0.0167236328125, 0.01434326171875, -0.038604736328125, -0.01038360595703125, -0.0105438232421875, -0.0191650390625, 0.0321044921875, 0.014556884765625, -0.021514892578125, 0.0160675048828125, -0.01186370849609375, 0.0013990402221679688, -0.004833221435546875, -0.033660888671875, -0.02581787109375, 0.04565429

## Create FAISS Vector Store

In [9]:
vector_store = FAISS.from_documents(
    documents = chunks,
    embedding = embeddings
)

print(f"Vector store created with {vector_store.index.ntotal} vectors.")

Vector store created with 4 vectors.


In [10]:
## Save vector store for later use
vector_store.save_local("data/.faiss_index")
print("Vector store saved locally.")

Vector store saved locally.


In [12]:
## load vector store from local
vector_store = FAISS.load_local(
    "data/.faiss_index",
    embeddings,
    allow_dangerous_deserialization = True
)

print(f"Vector store loaded with {vector_store.index.ntotal} vectors.")

Vector store loaded with 4 vectors.


In [ ]:
## Similarity Search
query = "What is deep learning?"

results = vector_store.similarity_search(query, k = 3)
print(results)

[Document(id='9e95a3a9-3ae4-4618-8322-9ba4996bbef5', metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recognition.'), Document(id='b9631ac0-5577-443c-bf2a-abf563d3584e', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'), Document(id='3b785e9f-37b5-4414-8a92-601f24ed986b', metadata={'source': 'NLP Overview', 'page': 1, 'topic': 'NLP'}, page_content='Natural Language Processing (NLP) is a branch of AI that helps computers understand human lang

In [14]:
print(f"Query: {query}\n")
print("Top 3 similar chunks:")
for i, doc in enumerate(results):
    print(f"\n{i+1}. Source: {doc.metadata['source']}")
    print(f"   Content: {doc.page_content[:200]}...")

Query: What is deep learning?

Top 3 similar chunks:

1. Source: Deep Learning
   Content: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning ...

2. Source: ML Basics
   Content: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised...

3. Source: NLP Overview
   Content: Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
      ...


In [15]:
## Similarity Search with score
result_with_scores = vector_store.similarity_search_with_score(query, k = 3)

print(f"Query: {query}\n")
print("Top 3 similar chunks with scores:")
for i, (doc, score) in enumerate(result_with_scores):
    print(f"\n{i+1}. Source: {doc.metadata['source']}")
    print(f"   Content: {doc.page_content[:200]}...")
    print(f"   Similarity Score: {score}")

Query: What is deep learning?

Top 3 similar chunks with scores:

1. Source: Deep Learning
   Content: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning ...
   Similarity Score: 0.5127016305923462

2. Source: ML Basics
   Content: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised...
   Similarity Score: 1.1548771858215332

3. Source: NLP Overview
   Content: Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
      ...
   Similarity Score: 1.2489064931869507


In [16]:
chunks

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.'),
 Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'),
 Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recogn

In [17]:
## Search with metadata filtering

filter_dict = { "topic": "ML" }
filtered_results = vector_store.similarity_search(
    query,
    k = 3,
    filter = filter_dict
)
print(filtered_results)

[Document(id='b9631ac0-5577-443c-bf2a-abf563d3584e', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.')]


In [20]:
print("Similarity Search Results:")
for i, result in enumerate(filtered_results):
    print(f"{i+1}. {result.page_content}")

Similarity Search Results:
1. Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.


## Build RAG Chain With LCEL

In [27]:
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    api_key = api_key,
    base_url = base_url
)
llm

ChatOpenAI(profile={'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002A5D20CC910>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002A5D20CCCD0>, root_client=<openai.OpenAI object at 0x000002A5D20CC690>, root_async_client=<openai.AsyncOpenAI object at 0x000002A5D20CCA50>, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.chatanywhere.org')

In [28]:
# 1. Simple RAG Chain with LCEL
simple_prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context:
Context: {context}

Question: {question}

Answer:""")

In [29]:
# Basic retriever
retriever = vector_store.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k": 3}
)
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002A594A4C910>, search_kwargs={'k': 3})

In [30]:
#  Format documents for the prompt
from typing import List
def format_docs(docs: List[Document]) -> str:
    """Format documents for insertion into the prompt."""
    formatted = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get("source", "Unknown")
        formatted.append(f"Document {i+1} ({source}): {doc.page_content}")
    return "\n\n".join(formatted)

In [39]:
simple_rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | simple_prompt
    | llm
    | StrOutputParser()
)
simple_rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002A594A4C910>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\nContext: {context}\n\nQuestion: {question}\n\nAnswer:'), additional_kwargs={})])
| ChatOpenAI(profile={'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'image_url_inp

In [40]:
## Conversational Rag Chain

conversational_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Use the provided context to answer questions."),
    ("placeholder", "{chat_history}"),
    ("human", "Context: {context}\n\nQuestion: {input}"),
])

conversational_prompt

ChatPromptTemplate(input_variables=['context', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMes

In [41]:
def create_conversational_rag():
    """Create a conversational RAG chain."""
    return (
        RunnablePassthrough.assign(
            context = lambda x: format_docs(retriever.invoke(x["input"]))
        )
        | conversational_prompt
        | llm
        | StrOutputParser()
    )

conversational_rag = create_conversational_rag()
conversational_rag

RunnableAssign(mapper={
  context: RunnableLambda(lambda x: format_docs(retriever.invoke(x['input'])))
})
| ChatPromptTemplate(input_variables=['context', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag=

In [42]:
### streaming RAG chain

streaming_rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | simple_prompt
    | llm
)

print("Modern RAG chains created successfully!")
print("Available chains:")
print("- simple_rag_chain: Basic Q&A")
print("- conversational_rag: Maintains conversation history")
print("- streaming_rag_chain: Supports token streaming")

Modern RAG chains created successfully!
Available chains:
- simple_rag_chain: Basic Q&A
- conversational_rag: Maintains conversation history
- streaming_rag_chain: Supports token streaming


In [46]:
# Test function for different chain types
def test_rag_chains(question: str):
    """Test all RAG chain variants"""
    print(f"Question: {question}")
    print("=" * 80)
    
    # 1. Simple RAG
    print("\n1. Simple RAG Chain:")
    answer = simple_rag_chain.invoke(question)
    print(f"Answer: {answer}")

    print("\n2. Streaming RAG:")
    print("Answer: ", end="", flush=True)
    for chunk in streaming_rag_chain.stream(question):
        print(chunk.content, end="", flush=True)
    print()

In [44]:
test_rag_chains("What is the difference between AI and machine learning")

Question: What is the difference between AI and machine learning

1. Simple RAG Chain:
Answer: The difference between AI and machine learning is that AI refers to the broader concept of simulating human intelligence in machines, while machine learning is a specific subset of AI that enables systems to learn from data by finding patterns, rather than being explicitly programmed.

2. Streaming RAG:
Answer: The difference between AI and machine learning is that AI (Artificial Intelligence) refers to the broader concept of simulating human intelligence in machines, allowing them to think and act like humans. In contrast, machine learning (ML) is a subset of AI that specifically focuses on enabling systems to learn from data by finding patterns without being explicitly programmed.


In [47]:
# Test with multiple questions
test_questions = [
    "What is the difference between AI and Machine Learning?",
    "Explain deep learning in simple terms",
    "How does NLP work?"
]

for question in test_questions:
    print("\n" + "=" * 80 + "\n")
    test_rag_chains(question)



Question: What is the difference between AI and Machine Learning?

1. Simple RAG Chain:
Answer: The difference between AI and Machine Learning is that AI refers to the simulation of human intelligence in machines, enabling them to think and act like humans. In contrast, Machine Learning is a subset of AI that specifically focuses on enabling systems to learn from data by finding patterns, rather than being explicitly programmed. While all Machine Learning is considered AI, not all AI involves Machine Learning.

2. Streaming RAG:
Answer: The difference between AI and Machine Learning is that AI (Artificial Intelligence) is the broader concept that refers to the simulation of human intelligence in machines, allowing them to think and act like humans. Machine Learning, on the other hand, is a subset of AI that focuses specifically on enabling systems to learn from data and find patterns without being explicitly programmed.


Question: Explain deep learning in simple terms

1. Simple RAG

In [48]:
## Conversational example
print("\n3. Conversational RAG:")
chat_history = []

query1 = "What is machine learning?"
answer1 = conversational_rag.invoke({
    "input": query1,
    "chat_history": chat_history
})

print(f"query: {query1}")
print(f"answer: {answer1}")


3. Conversational RAG:
query: What is machine learning?
answer: Machine Learning (ML) is a subset of Artificial Intelligence (AI) that enables systems to learn from data rather than being explicitly programmed. ML algorithms identify patterns in data and can be categorized into common types such as supervised learning, unsupervised learning, and reinforcement learning.


In [49]:
chat_history.extend([
    AIMessage(content=answer1),
    HumanMessage(content=query1)
])

In [50]:
query2 = "How is it different from traditional programming?"
answer2 = conversational_rag.invoke({
    "input": query2,
    "chat_history": chat_history
})

print(f"query: {query2}")
print(f"answer: {answer2}")

query: How is it different from traditional programming?
answer: Machine learning differs from traditional programming in that, instead of being explicitly programmed with specific rules and instructions to perform a task, machine learning systems learn from data. In traditional programming, a programmer writes code to solve a problem based on defined algorithms and logic. In contrast, machine learning algorithms automatically identify patterns and insights from large amounts of data, allowing the system to improve its performance over time without needing to be reprogrammed for each new scenario. This learning process enables machines to adapt and make decisions based on the data they encounter.
